[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yryo1005/OpenCampus_Demo/blob/main/OC_SpeechRecognition.ipynb)


# 音声認識デモ（Whisper）

マイク録音または音声ファイルから，**日本語／英語**の文字起こしを行うデモです．  
[OpenAI Whisper](https://github.com/openai/whisper)（Hugging Face Transformers）を Colab の GPU 上で動かします．

**実行環境**: Google Colab（ランタイム → GPU: T4 推奨）

## セルの進め方
1. **設定**（モデルサイズ・言語の既定値）
2. **ライブラリのインストール**
3. **ライブラリの読み込み・モデル準備・サンプル音声のダウンロード**
4. **Gradio の起動**

> API キーは **不要** です（推論はすべて Colab 内で完結）．  
> インストール直後にエラーが出る場合は，**ランタイム → セッションを再起動**してから設定セルとセル3以降を再実行してください．


## 0. 設定

- 認識精度を上げたい場合は `MODEL_ID` を大きいモデルに変更してください（VRAM・時間が増えます）．
- 変更後は **初期化セル** と **Gradio 起動セル** を再実行してください．


In [ ]:
# Whisper モデル（T4 / 14GB VRAM 向け）
# tiny < base < small < medium < large-v3（大きいほど精度↑・時間↑）
MODEL_ID = "openai/whisper-small"

# Gradio の言語セレクトの初期値
# "auto" = 自動判定 / "japanese" / "english"
DEFAULT_LANGUAGE = "auto"

print(f"MODEL_ID = {MODEL_ID}")
print(f"DEFAULT_LANGUAGE = {DEFAULT_LANGUAGE}")


## 1. ライブラリのインストール


In [ ]:
# Colab 標準の torch / transformers / gradio / soundfile / librosa を利用
# Whisper 用に transformers を念のため更新
!pip install -q -U "transformers>=4.40.0"


## 2. ライブラリの読み込み，変数のインスタンス化

サンプル音声（日本語・英語）をインターネットからダウンロードし，Whisper を読み込みます．  
初回はモデルのダウンロードに数分かかることがあります．


In [ ]:
from __future__ import annotations

import urllib.request
from pathlib import Path

import gradio as gr
import torch
from tqdm.auto import tqdm
from transformers import pipeline

# ------------------------------------------------------------
# 定数・サンプル音声 URL
# ------------------------------------------------------------
SAMPLE_DIR = Path("samples_speech")

# 短い公開サンプル（日本語: JSUT，英語: Whisper テスト用 JFK）
SAMPLE_AUDIO_SOURCES: list[tuple[str, str, str]] = [
    (
        "japanese_jsut.flac",
        "https://huggingface.co/datasets/japanese-asr/ja_asr.jsut_basic5000/resolve/main/sample.flac",
        "日本語（JSUT Basic5000 サンプル）",
    ),
    (
        "japanese_cv.flac",
        "https://huggingface.co/datasets/japanese-asr/ja_asr.common_voice_8_0/resolve/main/sample.flac",
        "日本語（Common Voice サンプル）",
    ),
    (
        "english_jfk.flac",
        "https://raw.githubusercontent.com/openai/whisper/main/tests/jfk.flac",
        "英語（JFK 演説の抜粋）",
    ),
]

LANGUAGE_CHOICES: list[tuple[str, str]] = [
    ("自動判定", "auto"),
    ("日本語", "japanese"),
    ("英語", "english"),
]


def resolve_device() -> str:
    """利用可能な推論デバイスを返す．

    Returns:
        str: "cuda" または "cpu"
    """
    if torch.cuda.is_available():
        name = torch.cuda.get_device_name(0)
        mem_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
        print(f"GPU: {name} ({mem_gb:.1f} GB)")
        return "cuda"
    print("GPU が見つかりません．CPU で実行します（時間がかかります）．")
    return "cpu"


def download_file(url: str, save_path: Path) -> Path:
    """URL からファイルをダウンロードする（既存ならスキップ）．

    Args:
        url (str): ダウンロード元 URL
        save_path (Path): 保存先パス

    Returns:
        Path: 保存したファイルのパス
    """
    if save_path.exists() and save_path.stat().st_size > 0:
        return save_path
    save_path.parent.mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve(url, save_path)
    return save_path


def prepare_sample_audios(
    sources: list[tuple[str, str, str]],
    sample_dir: Path,
) -> list[tuple[str, Path]]:
    """サンプル音声をダウンロードし，ラベルとパスの一覧を返す．

    Args:
        sources (list[tuple[str, str, str]]): (ファイル名, URL, 表示ラベル) のリスト
        sample_dir (Path): 保存先ディレクトリ

    Returns:
        list[tuple[str, Path]]: (表示ラベル, ローカルパス) のリスト
    """
    prepared: list[tuple[str, Path]] = []
    for filename, url, label in tqdm(sources, desc="サンプル音声DL", leave=False):
        path = download_file(url, sample_dir / filename)
        print(f"  {label}: {path} ({path.stat().st_size} bytes)")
        prepared.append((label, path))
    return prepared


def load_asr_pipeline(model_id: str, device: str):
    """Whisper の音声認識パイプラインを構築する．

    Args:
        model_id (str): Hugging Face モデル ID（例: openai/whisper-small）
        device (str): "cuda" または "cpu"

    Returns:
        transformers.pipelines.Pipeline: ASR パイプライン
    """
    dtype = torch.float16 if device == "cuda" else torch.float32
    device_index = 0 if device == "cuda" else -1
    print(f"モデルを読み込み中: {model_id} (dtype={dtype}, device={device})")
    asr = pipeline(
        task="automatic-speech-recognition",
        model=model_id,
        torch_dtype=dtype,
        device=device_index,
    )
    return asr


def transcribe_audio(
    audio_path: str | None,
    language: str = "auto",
) -> str:
    """音声ファイルを文字起こしする（Gradio コールバック）．

    Args:
        audio_path (str | None): 音声ファイルパス．未入力なら None
        language (str): "auto" / "japanese" / "english"

    Returns:
        str: 認識結果テキスト，またはエラーメッセージ
    """
    if not audio_path:
        return "音声をマイクで録音するか，ファイルをアップロードしてください．"

    generate_kwargs: dict = {"task": "transcribe"}
    if language and language != "auto":
        generate_kwargs["language"] = language

    # 短いクリップ想定．進捗はモデル内部の生成に依存
    for _ in tqdm(range(1), desc="音声認識", leave=False):
        result = asr_pipe(audio_path, generate_kwargs=generate_kwargs)

    text = (result.get("text") or "").strip()
    if not text:
        return "（認識結果が空でした．もう一度はっきり話してみてください．）"
    return text


def build_demo(sample_audios: list[tuple[str, Path]]) -> gr.Blocks:
    """Gradio UI を構築する．

    Args:
        sample_audios (list[tuple[str, Path]]): (ラベル, パス) のサンプル一覧

    Returns:
        gr.Blocks: デモ用 UI
    """
    with gr.Blocks(title="音声認識デモ（Whisper）") as demo:
        gr.Markdown(
            "## 音声を文字に変換\n"
            "日本語または英語で話してください．マイク録音・ファイルアップロード・サンプル再生に対応します．"
        )
        with gr.Row():
            with gr.Column(scale=1):
                audio_in = gr.Audio(
                    label="音声入力（マイク / アップロード）",
                    sources=["microphone", "upload"],
                    type="filepath",
                )
                language_in = gr.Radio(
                    choices=LANGUAGE_CHOICES,
                    value=DEFAULT_LANGUAGE,
                    label="言語",
                )
                run_btn = gr.Button("文字起こし", variant="primary")
            with gr.Column(scale=1):
                text_out = gr.Textbox(
                    label="認識結果",
                    lines=8,
                    interactive=False,
                )

        example_rows = [[str(path), "auto"] for _, path in sample_audios]
        gr.Examples(
            examples=example_rows,
            inputs=[audio_in, language_in],
            label="サンプル音声（クリックして入力欄に入れる）",
        )

        run_btn.click(
            fn=transcribe_audio,
            inputs=[audio_in, language_in],
            outputs=[text_out],
        )

    return demo


# ------------------------------------------------------------
# 初期化
# ------------------------------------------------------------
device = resolve_device()
sample_audios = prepare_sample_audios(SAMPLE_AUDIO_SOURCES, SAMPLE_DIR)

for _ in tqdm(range(1), desc="ASRパイプライン準備", leave=False):
    asr_pipe = load_asr_pipeline(MODEL_ID, device)

print("準備完了．次のセルで Gradio を起動してください．")


## 3. Gradio の実行

起動後，表示された UI または公開 URL から操作できます．


In [ ]:
demo = build_demo(sample_audios)
demo.launch(share=True)
